# NYC TLC Yellow Taxi — Exploratory Data Analysis
**Goal**: Understand the data deeply before modelling demand.

Sections:
1. Load & schema check
2. Volume over time
3. Missing values & data quality
4. Trip distance & fare distributions
5. Temporal patterns (hour / day / month)
6. Spatial patterns (top pickup zones)
7. Demand heatmap (zone × hour)
8. Feature correlations
9. Key takeaways for modelling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ROOT       = Path('..') 
YELLOW_DIR = ROOT / 'data' / 'raw' / 'yellow'
LOOKUP     = ROOT / 'Meta Data' / 'Lookups' / 'taxi_zone_lookup.csv'

print('Yellow files available:', len(list(YELLOW_DIR.glob('*.parquet'))))

## 1. Load & Schema Check

In [ ]:
# Load all parquet files — sample 20% per file to keep memory manageable
# Set SAMPLE_FRAC=1.0 to load everything if RAM allows
SAMPLE_FRAC = 0.20

files = sorted(YELLOW_DIR.glob('*.parquet'))
print(f'Loading {len(files)} files at {SAMPLE_FRAC*100:.0f}% sample...')

chunks = []
for f in files:
    df_chunk = pd.read_parquet(f)
    if SAMPLE_FRAC < 1.0:
        df_chunk = df_chunk.sample(frac=SAMPLE_FRAC, random_state=42)
    chunks.append(df_chunk)

df = pd.concat(chunks, ignore_index=True)
print(f'Total rows loaded: {len(df):,}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
df.dtypes

In [ ]:
df.head(3)

In [ ]:
df.describe()

## 2. Volume Over Time

In [ ]:
df['pickup_dt'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['year']      = df['pickup_dt'].dt.year
df['month']     = df['pickup_dt'].dt.month
df['yearmonth'] = df['pickup_dt'].dt.to_period('M')

monthly = df.groupby('yearmonth').size().reset_index(name='trips')
monthly['yearmonth_str'] = monthly['yearmonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly['yearmonth_str'], monthly['trips'] / 1e6, color='steelblue')
ax.set_title('Monthly Yellow Taxi Trip Volume (sampled)', fontsize=13)
ax.set_ylabel('Trips (millions)')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Missing Values & Data Quality

**Issues found:**

| Column | Missing | Root Cause |
|---|---|---|
| `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` | **14.13%** | Same block of rows — all have `payment_type=0` (unknown). Likely a vendor submission gap from Vendor 2. |
| `airport_fee` vs `Airport_fee` | Schema split | **Duplicate columns** from a mid-dataset rename. They never overlap — must be coalesced into one. |
| `cbd_congestion_fee` | **58.7%** | Introduced with NYC's CBD congestion pricing (Jan 5, 2025). Pre-policy rows are legitimately null → treat as $0. |

**Row-level garbage:**
- **2.98% negative fares** — meter corrections or test records
- **2.36% zero-distance trips** — likely cancelled or app-dispatched with no GPS
- **0.94% zero-passenger count** — driver entry errors
- **646 rows** where dropoff < pickup — clock errors
- **`RatecodeID=99`** (169K rows) — invalid code, not in data dictionary
- **`VendorID` 6 & 7** — non-standard, likely test/staging vendor IDs

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0')

In [ ]:
# Sanity checks
print('Negative fares:    ', (df['fare_amount'] < 0).sum())
print('Zero distance:     ', (df['trip_distance'] == 0).sum())
print('Passenger = 0:     ', (df['passenger_count'] == 0).sum())
print('Future pickups:    ', (df['pickup_dt'] > pd.Timestamp('2026-03-01')).sum())
print('Very old pickups:  ', (df['pickup_dt'] < pd.Timestamp('2022-01-01')).sum())

In [ ]:
# Clean: drop obvious garbage rows
before = len(df)
df = df[
    (df['fare_amount'] > 0) &
    (df['trip_distance'] > 0) &
    (df['pickup_dt'] >= '2023-01-01') &
    (df['pickup_dt'] <  '2026-03-01') &
    (df['PULocationID'].between(1, 263)) &
    (df['DOLocationID'].between(1, 263))
].copy()
print(f'Dropped {before - len(df):,} rows ({(before-len(df))/before*100:.2f}%)')
print(f'Clean rows: {len(df):,}')

## 4. Trip Distance & Fare Distributions

**Key findings:**
- **Fares** are right-skewed: median $13.58, mean $19.14 (mean pulled up by airport runs). 99th pct = $79.30. A small number of extreme outliers (>$500) exist — likely data entry errors.
- **Distance** is heavily right-skewed: median 1.80 mi, mean 5.38 mi (mean inflated by JFK/airport trips). The typical street hail is a short Manhattan hop.
- **Tips** (card payments only): median $3.28, mean $4.36. Cash tips are not captured at all — tip analysis is incomplete.
- The high mean/median gap on distance flags airport trips as a structurally different trip type — these should be handled carefully in the model (RatecodeID=2/3 = JFK/Newark).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fare amount
fare_cap = df['fare_amount'].quantile(0.99)
axes[0].hist(df.loc[df['fare_amount'] <= fare_cap, 'fare_amount'], bins=60, color='steelblue', edgecolor='none')
axes[0].set_title('Fare Amount (capped 99th pct)')
axes[0].set_xlabel('USD')

# Trip distance
dist_cap = df['trip_distance'].quantile(0.99)
axes[1].hist(df.loc[df['trip_distance'] <= dist_cap, 'trip_distance'], bins=60, color='coral', edgecolor='none')
axes[1].set_title('Trip Distance (capped 99th pct)')
axes[1].set_xlabel('Miles')

# Tip amount (credit card only)
tips = df.loc[(df['payment_type'] == 1) & (df['tip_amount'] > 0), 'tip_amount']
tip_cap = tips.quantile(0.99)
axes[2].hist(tips[tips <= tip_cap], bins=60, color='mediumseagreen', edgecolor='none')
axes[2].set_title('Tip Amount — Card Payments')
axes[2].set_xlabel('USD')

plt.tight_layout()
plt.show()

## 5. Temporal Patterns

**Key findings:**
- **Peak hours: 17, 18, 19** (evening rush). Trough hours: 3, 4, 5 (dead of night). Peak-to-trough ratio = **11x** — enormous swing, this is the #1 signal for the model.
- **Busiest day: Thursday** (2.11M trips in sample). **Quietest: Monday** (1.66M). Weekday > Weekend for yellow cabs — the inverse of rideshare.
- **Seasonal swing: 1.41x** (January peak, August trough). January being the highest is notable — likely driven by NYC's CBD congestion pricing launch (Jan 5, 2025) pushing some trips to yellow cabs. August is lowest — summer vacations + tourists using rideshare.
- The dual-peak pattern (morning commute + evening rush) is the dominant daily structure, but evening peak is stronger and longer.

In [ ]:
df['hour']       = df['pickup_dt'].dt.hour
df['dayofweek']  = df['pickup_dt'].dt.dayofweek   # 0=Mon, 6=Sun
df['dayname']    = df['pickup_dt'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Hourly demand
hourly = df.groupby('hour').size()
axes[0].bar(hourly.index, hourly.values / 1e6, color='steelblue')
axes[0].set_title('Trips by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Trips (millions)')
axes[0].set_xticks(range(0, 24))

# Day of week demand
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df.groupby('dayname').size().reindex(day_order)
axes[1].bar(daily.index, daily.values / 1e6, color='coral')
axes[1].set_title('Trips by Day of Week')
axes[1].set_xlabel('')
axes[1].set_ylabel('Trips (millions)')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Month of year — seasonality
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_demand = df.groupby('month').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([month_names[m-1] for m in monthly_demand.index], monthly_demand.values / 1e6, color='mediumpurple')
ax.set_title('Trips by Month (all years combined)')
ax.set_ylabel('Trips (millions)')
plt.tight_layout()
plt.show()

## 6. Spatial Patterns — Top Pickup Zones

**Key findings:**
- **Manhattan dominates: 87.3%** of all pickups. Queens is a distant second at 9.6%, driven almost entirely by JFK and LaGuardia airports.
- **Top 3 zones**: JFK Airport (1), Upper East Side South (2), Midtown Center (3). Airports are outliers — very different demand patterns (queue-based, not hail-based).
- **Brooklyn 2.2%, Bronx 0.5%, Staten Island ~0%** — yellow cabs overwhelmingly serve Manhattan. This is a structural market reality, not a data issue.
- For the routing product, **the airport zones need special treatment** — they are dispatch queue zones, not street hail zones. The model should flag these separately.

In [ ]:
zones = pd.read_csv(LOOKUP)
print(zones.shape)
zones.head()

In [ ]:
zone_demand = (
    df.groupby('PULocationID').size()
    .reset_index(name='trips')
    .merge(zones, left_on='PULocationID', right_on='LocationID')
    .sort_values('trips', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 5))
top20 = zone_demand.head(20)
ax.barh(top20['Zone'], top20['trips'] / 1e6, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 20 Pickup Zones')
ax.set_xlabel('Trips (millions)')
plt.tight_layout()
plt.show()

In [ ]:
# Demand by borough
borough_demand = (
    df.merge(zones[['LocationID','Borough']], left_on='PULocationID', right_on='LocationID')
    .groupby('Borough').size()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(borough_demand.index, borough_demand.values / 1e6, color='teal')
ax.set_title('Trips by Borough (Pickup)')
ax.set_ylabel('Trips (millions)')
plt.tight_layout()
plt.show()

## 7. Demand Heatmap — Zone × Hour (Top 30 Zones)

**Key findings:**
- **Airport zones (JFK, LaGuardia)** show a flat or afternoon-skewed profile — driven by flight schedules, not commuting. Very different from Manhattan zones.
- **Midtown zones** (Times Square, Penn Station, Midtown Center) show a classic double-peak: morning commute (8–9am) + strong evening (5–8pm).
- **Upper East/West Side** residential zones peak sharply in the **morning outbound** (7–9am) and evening inbound — classic commuter pattern.
- **Penn Station/Madison Sq West** has a distinctive late-night bump (10pm–midnight) — entertainment + transit hub.
- The heatmap reveals that no two zones are the same — **zone ID alone is a strong feature** for the model, justifying zone-level embeddings rather than just borough.

In [ ]:
top30_zones = zone_demand.head(30)['PULocationID'].tolist()

heatmap_data = (
    df[df['PULocationID'].isin(top30_zones)]
    .groupby(['PULocationID', 'hour']).size()
    .unstack(fill_value=0)
)

# Add zone names as index
zone_name_map = zones.set_index('LocationID')['Zone'].to_dict()
heatmap_data.index = heatmap_data.index.map(zone_name_map)

# Normalise each row (zone) to show within-zone hourly pattern
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_norm,
    cmap='YlOrRd',
    ax=ax,
    linewidths=0.3,
    cbar_kws={'label': 'Share of daily trips'}
)
ax.set_title('Hourly Demand Pattern by Zone (normalised)', fontsize=13)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 8. Feature Correlations

In [ ]:
corr_cols = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
             'passenger_count', 'hour', 'dayofweek']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 9. Demand Aggregation — Build the Modelling Target

Aggregate pickups to: **zone × 15-minute interval → trip count**

**Important note on sparsity**: This table only contains (zone, slot) pairs where ≥1 trip occurred. After aggregation, **33.8% of all zone-slot combinations have ≤1 trip**, and many valid (zone, time) combinations are simply absent (implying 0 trips). The cleaning notebook zero-fills these gaps before modelling — failing to do so causes the model to never learn "quiet zone at 4am = 0 trips".

In [ ]:
df['time_bucket'] = df['pickup_dt'].dt.floor('15min')

demand = (
    df.groupby(['PULocationID', 'time_bucket'])
    .size()
    .reset_index(name='trip_count')
)

print(f'Demand table rows: {len(demand):,}')
print(f'Unique zones:      {demand["PULocationID"].nunique()}')
print(f'Unique time slots: {demand["time_bucket"].nunique()}')
demand.head()

In [ ]:
# Distribution of trip counts per 15-min slot across all zones
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(demand['trip_count'], bins=60, color='steelblue', edgecolor='none', log=True)
ax.set_title('Distribution of Trip Count per Zone per 15-min Slot (log scale)')
ax.set_xlabel('Trip count')
ax.set_ylabel('Frequency (log)')
plt.tight_layout()
plt.show()

print(demand['trip_count'].describe())

In [ ]:
# Save the demand table for use in modelling notebooks
out_path = ROOT / 'data' / 'processed' / 'demand_15min.parquet'
demand.to_parquet(out_path, index=False)
print(f'Saved to {out_path}')

## Key Takeaways for Modelling

### Data Quality Issues → Cleaning Required (see `02b_cleaning_and_fe.ipynb`)
| Issue | Fix |
|---|---|
| `airport_fee` / `Airport_fee` duplicate columns | Coalesce into single `airport_fee` |
| `cbd_congestion_fee` 58.7% null | Fill with 0 (pre-policy = no fee) |
| 14.13% block missing (`payment_type=0`) | Drop these rows — submission gap, not recoverable |
| Negative fares (2.98%) | Drop |
| Zero-distance trips (2.36%) | Drop |
| Zero-passenger count (0.94%) | Drop |
| Dropoff < pickup (646 rows) | Drop |
| `RatecodeID=99` | Drop |
| Extreme fares >$500 / distance >100mi | Cap at 99th percentile |

### Feature Engineering Required
| Feature | Why |
|---|---|
| **Zero-fill demand table** | 33.8% sparsity — missing slots = 0, not absent |
| **Congestion pricing flag** | Step-change from Jan 5, 2025 — structural demand shift |
| **Airport zone flag** | JFK/LaGuardia are queue zones, not street hails |
| **Holiday flags** | Major US holidays show demand drops |
| **Borough & service zone** | Richer than raw zone ID alone |
| **Lag features** | Same slot 1h/1day/1week ago — strongest model signals |

### Modelling Signals
- **Peak demand hours**: 17:00–19:00 (11x stronger than 3–5am)
- **Busiest day**: Thursday; quietest: Monday
- **Seasonality**: 1.41x swing (January peak, August trough)
- **Top street-hail zones**: Upper East Side South, Midtown Center, Midtown East (exclude airports from hail model)
- **Manhattan = 87.3%** of volume — model is effectively a Manhattan model
- **Target variable**: Right-skewed count data → use Poisson objective in LightGBM, not MSE